# Part 9 — Arenas: from Pairwise Battles to One Ranking

*Evals from First Principles, Part 9.*

A leaderboard turns thousands of pairwise "which answer is better?" battles into one ranking. We build it two ways by hand on the same four chatbots (A, B, C, D), each fed the same 18 head-to-head battles (a round-robin where every pair meets 3 times):

1. **Elo** — an *online* update after every single battle, using the same win probability formula chess ratings use. We replay the exact same 18 battles in two different orders and watch the final ratings disagree: Elo is **order-dependent**.
2. **Bradley-Terry** — fit *all* battles at once by maximum likelihood, using only the aggregate win matrix. Its ranking is **order-independent** by construction.

Pure Python, offline, deterministic. Any shuffle uses a fixed seed.

In [ ]:
import random

MODELS = ["A", "B", "C", "D"]

## Step 1 — The battle record

Four chatbots fought a round-robin: every pair battled 3 times, 18 battles total. `WINS[i][j]` is the number of times model `i` beat model `j` (a "win" = the judge preferred `i`). Reading a row tells you who `i` beat; reading a column tells you who beat `i`.

The true strength order baked in is A > B > C > D, with a couple of upsets: B beat A once, C beat B once, D beat C once. Those upsets are what will make Elo's answer depend on order.

In [ ]:
# Read across a row to see who i beat; down a column to see who beat i.
#         beat->   A  B  C  D
WINS = [
    [0, 2, 3, 3],   # A beat B twice, C 3x, D 3x   (8 wins, 1 loss)
    [1, 0, 2, 3],   # B beat A once, C 2x, D 3x    (6 wins, 3 losses)
    [0, 1, 0, 2],   # C beat B once, D 2x          (3 wins, 6 losses)
    [0, 0, 1, 0],   # D beat C once                (1 win,  8 losses)
]

## Step 2 — Elo: expected score and one update

Elo predicts a win probability from the rating gap alone: `E_A = 1/(1+10^((Rb-Ra)/400))`. After the battle resolves, both ratings move toward the surprise: `R' = R + K*(S - E)`, where `S` is 1 for a win and 0 for a loss, and `K` controls how fast ratings move.

We do one update by hand first: A beats B, both start at 1000, K=32. Equal ratings mean `E_A = 0.50`, a coin flip, so the win moves the full `K*0.5 = 16` points each way.

In [ ]:
def expected(ra, rb):
    """Probability the A-side scores, given ratings ra and rb (logistic, /400)."""
    return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))


def elo_update(ra, rb, score_a, k=32):
    """One battle. score_a is 1 if A won, 0 if A lost. Return (ra', rb')."""
    ea = expected(ra, rb)
    ra_new = ra + k * (score_a - ea)
    rb_new = rb + k * ((1 - score_a) - (1 - ea))
    return ra_new, rb_new

## Step 3 — Replay all 18 battles, in two different orders

To run Elo over the whole arena we first flatten the win matrix into a flat list of `(winner, loser)` battles, then replay that list against a fresh set of 1000-point ratings, one `elo_update` per battle.

We do this twice: once in the as-listed order the win matrix produces, and once after shuffling that same list of 18 battles with a fixed seed (`random.Random(0)`). Same 18 outcomes, same starting ratings, same K — only the order changes.

In [ ]:
def battle_log():
    """Flatten the win matrix into a list of (winner, loser) battles."""
    games = []
    for i in range(len(MODELS)):
        for j in range(len(MODELS)):
            for _ in range(WINS[i][j]):
                games.append((i, j))
    return games


def run_elo(games, k=32, start=1000.0):
    """Replay a sequence of (winner, loser) battles; return final ratings."""
    r = [start] * len(MODELS)
    for w, l in games:
        rw, rl = elo_update(r[w], r[l], 1.0, k)
        r[w], r[l] = rw, rl
    return r

## Step 4 — Bradley-Terry: fit all battles at once

Bradley-Terry treats the whole arena as one maximum-likelihood problem: find strengths `p` (one per model, normalized to sum to 1) such that `P(i beats j) = p_i / (p_i + p_j)` best explains the *aggregate* win matrix. We fit it with the classic Zermelo (1929) MM iteration:

`p_i <- W_i / sum_j n_ij/(p_i+p_j)`, renormalized to sum to 1 after every step, where `W_i` is model `i`'s total wins and `n_ij` the number of games between `i` and `j`.

Because this update only ever looks at the aggregate counts `W_i` and `n_ij`, never at the order battles happened in, order cannot matter to the result.

In [ ]:
def bt_fit(wins, iters=20):
    """
    Fit strengths p so that P(i beats j) = p_i / (p_i + p_j) is most likely.
    Update rule (Zermelo 1929): p_i <- W_i / sum_j n_ij/(p_i+p_j), where W_i is
    i's total wins and n_ij the games between i and j. Normalize sum(p)=1 each
    step. Depends ONLY on the aggregate matrix -> order cannot matter.
    Returns (final_p, snapshots) where snapshots are p after selected iters.
    """
    n = len(wins)
    total_wins = [sum(wins[i]) for i in range(n)]
    games_between = [[wins[i][j] + wins[j][i] for j in range(n)] for i in range(n)]
    p = [1.0 / n] * n
    snapshots = {0: p[:]}
    for step in range(1, iters + 1):
        new = [0.0] * n
        for i in range(n):
            denom = 0.0
            for j in range(n):
                if j != i:
                    denom += games_between[i][j] / (p[i] + p[j])
            new[i] = total_wins[i] / denom
        s = sum(new)
        p = [x / s for x in new]
        if step in (1, 2, 3, 20):
            snapshots[step] = p[:]
    return p, snapshots


def ranking(scores):
    """Return model letters ordered best-first by score."""
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return " > ".join(MODELS[i] for i in order)

## Step 5 — Run everything and print the comparison

Now we run the one-update example, replay all 18 battles in both orders through Elo, then fit Bradley-Terry on the same win matrix and watch its strengths converge over 20 iterations. The final printout puts Elo's two disagreeing rankings side by side with Bradley-Terry's single, order-independent one.

In [ ]:
line = "=" * 72
print(line)
print("PART 9 - ARENAS: from pairwise battles to one ranking.")
print(line)

# -- One Elo update, fully by hand --------------------------------------
print("ONE ELO UPDATE (A beats B, both start at 1000, K=32):")
ea = expected(1000.0, 1000.0)
print(f"  E_A = 1/(1+10^((Rb-Ra)/400)) = 1/(1+10^0) = {ea:.2f}")
print(f"  A won, so S_A = 1.  R_A' = 1000 + 32*(1 - {ea:.2f}) "
      f"= {1000 + 32 * (1 - ea):.1f}")
print(f"           S_B = 0.  R_B' = 1000 + 32*(0 - {ea:.2f}) "
      f"= {1000 + 32 * (0 - (1 - ea)):.1f}")
print("  Equal ratings -> a coin-flip prediction -> the win moves 16 points.")

# -- Elo is order-dependent --------------------------------------------
games = battle_log()
print("\n" + "-" * 72)
print(f"ELO OVER ALL {len(games)} BATTLES (start 1000, K=32).")
print("Same battles, two orders:")
print("-" * 72)

r_asplayed = run_elo(games)
shuffled = games[:]
random.Random(0).shuffle(shuffled)
r_shuffled = run_elo(shuffled)

print("  as-listed order:  " +
      "  ".join(f"{m}={r:.1f}" for m, r in zip(MODELS, r_asplayed)))
print("  shuffled order:   " +
      "  ".join(f"{m}={r:.1f}" for m, r in zip(MODELS, r_shuffled)))
diffs = [b - a for a, b in zip(r_asplayed, r_shuffled)]
print("  difference:       " +
      "  ".join(f"{m}={d:+.1f}" for m, d in zip(MODELS, diffs)))
print(f"  ranking (as-listed): {ranking(r_asplayed)}")
print(f"  ranking (shuffled):  {ranking(r_shuffled)}")
print("  -> Same battles, different NUMBERS: Elo depends on match order.")

# -- Bradley-Terry fits everything at once -----------------------------
print("\n" + "-" * 72)
print("BRADLEY-TERRY MLE on the aggregate win matrix (iterated):")
print("-" * 72)
print("  win matrix (row beat col):   A  B  C  D")
for i, m in enumerate(MODELS):
    print(f"                          {m}  [ " +
          "  ".join(str(WINS[i][j]) for j in range(len(MODELS))) + " ]")

p, snaps = bt_fit(WINS)
print("\n  strengths p (normalized to sum 1), by iteration:")
print("                    A       B       C       D")
for step in (0, 1, 2, 3, 20):
    row = snaps[step]
    tag = "start" if step == 0 else f"it {step}"
    print(f"    {tag:>6}      " + "  ".join(f"{x:.4f}" for x in row))
print(f"\n  converged ranking: {ranking(p)}")
print(f"  P(A beats D) = pA/(pA+pD) = {p[0] / (p[0] + p[3]):.3f}")
print(f"  P(B beats C) = pB/(pB+pC) = {p[1] / (p[1] + p[2]):.3f}")
print("  -> No order anywhere: BT reads only the totals, so its ranking is")
print("     the same no matter how the battles were shuffled.")
print(line)

## Recap

- Both methods are fed the exact same 18 battles between the exact same four chatbots. The only thing that differs is *how* they consume the data.
- **Elo** updates online, one battle at a time, so early battles set ratings that later battles nudge — the final numbers (and even the ranking, in noisier arenas) depend on the order battles are replayed in.
- **Bradley-Terry** fits *all* battles jointly against the aggregate win matrix by maximum likelihood, so it has no notion of order at all: shuffle the battle log a thousand ways and you get the identical strengths every time.
- Both methods agreed here (A > B > C > D), because the upsets were mild. In a live arena with thousands of battles arriving in whatever order users happen to submit them, that agreement is not guaranteed, which is why serious leaderboards report Bradley-Terry (or an order-averaged Elo) rather than raw sequential Elo.

Next: annotator agreement, how to tell whether the humans (or judges) generating your labels agree with each other in the first place.